# Amazon SageMaker Asynchronous Inference


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

---

_**A new near real-time Inference option for generating machine learning model predictions**_

**Table of Contents**

* [Background](#background)
* [Notebook Scope](#scope)
* [Overview and sample end to end flow](#overview)
* [Section 1 - Setup](#setup) 
    * [Create Model](#createmodel)
    * [Create EndpointConfig](#endpoint-config)
    * [Create Endpoint](#create-endpoint)
    * [Setup AutoScaling policy (Optional)](#setup-autoscaling)
* [Section 2 - Using the Endpoint](#endpoint) 
    * [Invoke Endpoint](#invoke-endpoint)
    * [Check Output Location](#check-output)
    * [Multiple Invocations](#multiple-invoke)  
* [Section 3 - Clean up](#clean)

### Background <a id='background'></a>  
Amazon SageMaker Asynchronous Inference is a new capability in SageMaker that queues incoming requests and processes them asynchronously. SageMaker currently offers two inference options for customers to deploy machine learning models: 1) a real-time option for low-latency workloads 2) Batch transform, an offline option to process inference requests on batches of data available upfront. Real-time inference is suited for workloads with payload sizes of less than 6 MB and require inference requests to be processed within 60 seconds. Batch transform is suitable for offline inference on batches of data. 

Asynchronous inference is a new inference option for near real-time inference needs. Requests can take up to 15 minutes to process and have payload sizes of up to 1 GB. Asynchronous inference is suitable for workloads that do not have sub-second latency requirements and have relaxed latency requirements. For example, you might need to process an inference on a large image of several MBs within 5 minutes. In addition, asynchronous inference endpoints let you control costs by scaling down endpoints instance count to zero when they are idle, so you only pay when your endpoints are processing requests. 

### Notebook scope <a id='scope'></a>  
This notebook provides an introduction to the SageMaker Asynchronous inference capability. This notebook will cover the steps required to create an asynchronous inference endpoint and test it with some sample requests. 

### Overview and sample end to end flow <a id='overview'></a>
Asynchronous inference endpoints have many similarities (and some key differences) compared to real-time endpoints. The process to create asynchronous endpoints is similar to real-time endpoints. You need to create: a model, an endpoint configuration, and then an endpoint. However, there are specific configuration parameters specific to asynchronous inference endpoints which we will explore below. 

Invocation of asynchronous endpoints differ from real-time endpoints. Rather than pass request payload inline with the request, you upload the payload to Amazon S3 and pass an Amazon S3 URI as a part of the request. Upon receiving the request, SageMaker provides you with a token with the output location where the result will be placed once processed. Internally, SageMaker maintains a queue with these requests and processes them. During endpoint creation, you can optionally specify an Amazon SNS topic to receive success or error notifications. Once you receive the notification that your inference request has been successfully processed, you can access the result in the output Amazon S3 location. 

The diagram below provides a visual overview of the end-to-end flow with Asynchronous inference endpoint.

![title](images/e2e.png)

## Dataset

We're about to work with the [Titanic dataset](https://www.openml.org/d/40945)[1]. From the dataset documentation:

> The original Titanic dataset, describing the survival status of individual passengers on the Titanic. The titanic data does not contain information from the crew, but it does contain actual ages of half of the passengers. The principal source for data about Titanic passengers is the Encyclopedia Titanica. The datasets used here were begun by a variety of researchers. One of the original sources is Eaton & Haas (1994) Titanic: Triumph and Tragedy, Patrick Stephens Ltd, which includes a passenger list created by many researchers and edited by Michael A. Findlay.
>
> Thomas Cason of UVa has greatly updated and improved the Titanic data frame using the Encyclopedia Titanica and created the dataset here. Some duplicate passengers have been dropped, many errors corrected, many missing ages filled in, and new variables created.


---
## 1. Setup <a id='setup'></a>

First we ensure we have an updated version of boto3, which includes the latest SageMaker features:

Import the required python libraries:

In [1]:
# [exec-copy] pip install disabled for local papermill run
pass

In [2]:
import boto3
from time import gmtime, strftime
from datetime import datetime

# V3: SageMaker Session and get_execution_role now live in sagemaker-core
from sagemaker.core.helper.session_helper import Session, get_execution_role

boto_session = boto3.session.Session()
sm_session = Session(boto_session=boto_session)
sm_client = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")
region = boto_session.region_name

[07/16/26 09:52:48] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5527091;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5527092;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


Specify your IAM role. Go the AWS IAM console (https://console.aws.amazon.com/iam/home) and add the following policies to your IAM Role:

   * SageMakerFullAccessPolicy


   * (Optional) Amazon SNS access: Add `sns:Publish` on the topics you define. Apply this if you plan to use Amazon SNS to receive notifications.

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "sns:Publish"
            ],
            "Effect": "Allow",
            "Resource": "arn:aws:sns:<aws-region>:<account-id>:<topic-name>"
        }
    ]
}
```

* (Optional) KMS decrypt, encrypt if your Amazon S3 bucket is encrypte.

In [3]:
# Download the Input files and model from S3 bucket
!mkdir input
!mkdir model
s3 = boto3.client("s3")
for key in s3.list_objects(
    Bucket=f"sagemaker-example-files-prod-{region}", Prefix="models/async-inference/input-files/"
)["Contents"]:
    s3.download_file(
        f"sagemaker-example-files-prod-{region}", key["Key"], "input/" + key["Key"].split("/")[-1]
    )
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    "models/async-inference/demo-xgboost-model.tar.gz",
    "model/demo-xgboost-model.tar.gz",
)

[07/16/26 09:52:49] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5527097;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5527098;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Specify your SageMaker IAM Role (`sm_role`) and Amazon S3 bucket (`s3_bucket`). You can optionally use a default SageMaker Session IAM Role and Amazon S3 bucket. Make sure the role you use has the necessary permissions for SageMaker, Amazon S3, and optionally Amazon SNS.

In [4]:
sm_role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"  # [exec-copy] explicit role for local papermill run
# Feel free to use your own role here
# sm_role = "arn:aws:iam::123456789012:role/sagemaker-custom-role"
print(f"Using Role: {sm_role}")
s3_bucket = sm_session.default_bucket()
default_bucket_prefix = sm_session.default_bucket_prefix
print(f"Will use bucket '{s3_bucket}' for storing all resources related to this notebook")

Using Role: arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045


Will use bucket 'sagemaker-us-west-1-729646638167' for storing all resources related to this notebook


In [5]:
bucket_prefix = "async-inference-demo"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    bucket_prefix = f"{default_bucket_prefix}/{bucket_prefix}"

resource_name = "AsyncInferenceDemo-{}-{}"

Next, you will create a model with `CreateModel`, an endpoint configuration with `CreateEndpointConfig`, and then an endpoint with the `CreateEndpoint` API.


### 1.1 Create Model <a id='createmodel'></a>
Specify the location of the pre-trained model stored in Amazon S3. This example uses a pre-trained XGBoost model name demo-xgboost-model.tar.gz. The full Amazon S3 URI is stored in a string variable `model_url`. 

In [6]:
model_s3_key = f"{bucket_prefix}/demo-xgboost-model.tar.gz"
model_url = f"s3://{s3_bucket}/{model_s3_key}"
print(f"Uploading Model to {model_url}")

with open("model/demo-xgboost-model.tar.gz", "rb") as model_file:
    boto_session.resource("s3").Bucket(s3_bucket).Object(model_s3_key).upload_fileobj(model_file)

Uploading Model to s3://sagemaker-us-west-1-729646638167/async-inference-demo/demo-xgboost-model.tar.gz


Specify a primary container. For the primary container, you specify the Docker image that contains inference code, artifacts (from prior training), and a custom environment map that the inference code uses when you deploy the model for predictions. In this example, we specify an XGBoost built-in algorithm container image.

In [7]:
# V3: image_uris now lives in sagemaker-core
from sagemaker.core import image_uris

# Specify an AWS container image and region as desired
container = image_uris.retrieve(region=region, framework="xgboost", version="0.90-1")

[07/16/26 09:52:55] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5527103;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5527104;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/16/26 09:52:56] INFO     Defaulting to only available Python version: py3                     ]8;id=5527111;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=5527112;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: cpu.                       ]8;id=5527118;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=5527119;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#539\539]8;;\

Create a model by specifying the `ModelName`, the `ExecutionRoleARN` (the ARN of the IAM role that Amazon SageMaker can assume to access model artifacts/ docker images for deployment), and the `PrimaryContainer`.

In [8]:
model_name = resource_name.format("Model", datetime.now().strftime("%Y-%m-%d-%H-%M-%S"))
create_model_response = sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=sm_role,
    PrimaryContainer={
        "Image": container,
        "ModelDataUrl": model_url,
    },
)

print(f"Created Model: {create_model_response['ModelArn']}")

Created Model: arn:aws:sagemaker:us-west-1:729646638167:model/AsyncInferenceDemo-Model-2026-07-16-09-52-56


### 1.2 Create EndpointConfig <a id='endpointconfig'></a>

Once you have a model, create an endpoint configuration with `CreateEndpointConfig`. Amazon SageMaker hosting services uses this configuration to deploy models. In the configuration, you identify one or more model that were created using with `CreateModel` API, to deploy the resources that you want Amazon SageMaker to provision. Specify the `AsyncInferenceConfig` object and provide an output Amazon S3 location for `OutputConfig`. You can optionally specify Amazon SNS topics on which to send notifications about prediction results.

In [9]:
endpoint_config_name = resource_name.format(
    "EndpointConfig", datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
)
create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "ModelName": model_name,
            "InstanceType": "ml.m5.xlarge",
            "InitialInstanceCount": 1,
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {
            "S3OutputPath": f"s3://{s3_bucket}/{bucket_prefix}/output",
            # Optionally specify Amazon SNS topics
            # "NotificationConfig": {
            # "SuccessTopic": "arn:aws:sns:<aws-region>:<account-id>:<topic-name>",
            # "ErrorTopic": "arn:aws:sns:<aws-region>:<account-id>:<topic-name>",
            # }
        },
        "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 4},
    },
)
print(f"Created EndpointConfig: {create_endpoint_config_response['EndpointConfigArn']}")

Created EndpointConfig: arn:aws:sagemaker:us-west-1:729646638167:endpoint-config/AsyncInferenceDemo-EndpointConfig-2026-07-16-09-52-56


### 1.3 Create Endpoint <a id='create-endpoint'></a>

Once you have your model and endpoint configuration, use the `CreateEndpoint` API to create your endpoint. The endpoint name must be unique within an AWS Region in your AWS account.

In [10]:
endpoint_name = resource_name.format("Endpoint", datetime.now().strftime("%Y-%m-%d-%H-%M-%S"))

create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)
print(f"Created Endpoint: {create_endpoint_response['EndpointArn']}")

Created Endpoint: arn:aws:sagemaker:us-west-1:729646638167:endpoint/AsyncInferenceDemo-Endpoint-2026-07-16-09-52-56


Validate that the endpoint is created before invoking it:

In [11]:
waiter = sm_client.get_waiter("endpoint_in_service")
print("Waiting for endpoint to create...")
waiter.wait(EndpointName=endpoint_name)
resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
print(f"Endpoint Status: {resp['EndpointStatus']}")

Waiting for endpoint to create...


Endpoint Status: InService


### 1.4 Setup AutoScaling policy (Optional)    <a id='setup-autoscaling'></a>

This section describes how to configure autoscaling on your asynchronous endpoint using Application Autoscaling. You need to first register your endpoint variant with Application Autoscaling, define a scaling policy, and then apply the scaling policy. In this configuration, we use a custom metric, `CustomizedMetricSpecification`, called `ApproximateBacklogSizePerInstance`. Please refer to the SageMaker Developer guide for a detailed list of metrics available with your asynchronous inference endpoint.

In [12]:
client = boto3.client(
    "application-autoscaling"
)  # Common class representing Application Auto Scaling for SageMaker amongst other services

resource_id = (
    "endpoint/" + endpoint_name + "/variant/" + "variant1"
)  # This is the format in which application autoscaling references the endpoint

# Configure Autoscaling on asynchronous endpoint down to zero instances
response = client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=5,
)

response = client.put_scaling_policy(
    PolicyName="Invocations-ScalingPolicy",
    ServiceNamespace="sagemaker",  # The namespace of the AWS service that provides the resource.
    ResourceId=resource_id,  # Endpoint name
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",  # SageMaker supports only Instance Count
    PolicyType="TargetTrackingScaling",  # 'StepScaling'|'TargetTrackingScaling'
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # The target value for the metric. - here the metric is - SageMakerVariantInvocationsPerInstance
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": endpoint_name}],
            "Statistic": "Average",
        },
        "ScaleInCooldown": 600,  # The cooldown period helps you prevent your Auto Scaling group from launching or terminating
        # additional instances before the effects of previous activities are visible.
        # You can configure the length of time based on your instance startup time or other application needs.
        # ScaleInCooldown - The amount of time, in seconds, after a scale in activity completes before another scale in activity can start.
        "ScaleOutCooldown": 300,  # ScaleOutCooldown - The amount of time, in seconds, after a scale out activity completes before another scale out activity can start.
        # 'DisableScaleIn': True|False - ndicates whether scale in by the target tracking policy is disabled.
        # If the value is true , scale in is disabled and the target tracking policy won't remove capacity from the scalable resource.
    },
)

The endpoint is now ready for invocation.

--- 
## 2. Using the Endpoint <a id='endpoint'></a>

### 2.1 Uploading the Request Payload <a id='upload'></a>

First, you need to upload the request to Amazon S3. We define a function called, `upload_file`, to make it easier to make multiple invocations in a later step.

In [13]:
import os


def upload_file(input_location):
    prefix = f"{bucket_prefix}/input"
    return sm_session.upload_data(
        input_location,
        bucket=sm_session.default_bucket(),
        key_prefix=prefix,
        extra_args={"ContentType": "text/libsvm"},
    )

In [14]:
input_1_location = "input/test_point_0.libsvm"
input_1_s3_location = upload_file(input_1_location)

### 2.1 Invoke Endpoint   <a id='invoke-endpoint'></a>

Get inferences from the model hosted at your asynchronous endpoint with `InvokeEndpointAsync`. Specify the location of your inference data in the `InputLocation` field and the name of your endpoint for `EndpointName`. The response payload contains the output Amazon S3 location where the result will be placed. 

In [15]:
response = sm_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name, InputLocation=input_1_s3_location
)
output_location = response["OutputLocation"]
print(f"OutputLocation: {output_location}")

OutputLocation: s3://sagemaker-us-west-1-729646638167/async-inference-demo/output/a13f1e16-e86c-4571-bff9-1321a33a56fb.out


### 2.2 Check Output Location <a id='check-output'></a>

Check the output location to see if the inference has been processed. We make multiple requests (beginning of the `while True` statement in the `get_output` function) every two seconds until there is an output of the inference request: 

In [16]:
import urllib, time
from botocore.exceptions import ClientError


def get_output(output_location):
    output_url = urllib.parse.urlparse(output_location)
    bucket = output_url.netloc
    key = output_url.path[1:]
    while True:
        try:
            return sm_session.read_s3_file(bucket=output_url.netloc, key_prefix=output_url.path[1:])
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchKey":
                print("waiting for output...")
                time.sleep(2)
                continue
            raise

In [17]:
output = get_output(output_location)
print(f"Output: {output}")

waiting for output...


waiting for output...


Output: 0.809421181678772


### 2.3 Multiple Invocations  <a id='multiple-invoke'></a>

The following shows how you can invoke the endpoint with multiple requests:

In [18]:
inferences = []
for i in range(25):
    input_file = f"input/test_point_{i}.libsvm"
    input_file_s3_location = upload_file(input_file)
    print(f"Invoking Endpoint with {input_file}")
    response = sm_runtime.invoke_endpoint_async(
        EndpointName=endpoint_name, InputLocation=input_file_s3_location
    )
    output_location = response["OutputLocation"]
    inferences += [(input_file, output_location)]
    time.sleep(0.5)

for input_file, output_location in inferences:
    output = get_output(output_location)
    print(f"Input File: {input_file}, Output: {output}")

Invoking Endpoint with input/test_point_0.libsvm


Invoking Endpoint with input/test_point_1.libsvm


Invoking Endpoint with input/test_point_2.libsvm


Invoking Endpoint with input/test_point_3.libsvm


Invoking Endpoint with input/test_point_4.libsvm


...

Input File: input/test_point_9.libsvm, Output: 0.6018468141555786
Input File: input/test_point_10.libsvm, Output: 0.795388400554657
Input File: input/test_point_11.libsvm, Output: 0.3886236250400543


Input File: input/test_point_12.libsvm, Output: 0.9560881853103638
Input File: input/test_point_13.libsvm, Output: 0.3886236250400543
Input File: input/test_point_14.libsvm, Output: 0.5001285076141357
Input File: input/test_point_15.libsvm, Output: 0.45026054978370667


Input File: input/test_point_16.libsvm, Output: 0.8613329529762268
Input File: input/test_point_17.libsvm, Output: 0.5521252751350403
Input File: input/test_point_18.libsvm, Output: 0.6051725149154663
Input File: input/test_point_19.libsvm, Output: 0.5196762681007385


Input File: input/test_point_20.libsvm, Output: 0.3886236250400543
Input File: input/test_point_21.libsvm, Output: 0.42019087076187134


Input File: input/test_point_22.libsvm, Output: 0.4293918311595917
Input File: input/test_point_23.libsvm, Output: 0.3053364157676697
Input File: input/test_point_24.libsvm, Output: 0.9731993079185486


### 3. Clean up <a id='clean'></a>

If you enabled auto-scaling for your endpoint, ensure you deregister the endpoint as a scalable target before deleting the endpoint. To do this, run the following:

In [19]:
response = client.deregister_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
)

Remember to delete your endpoint after use as you will be charged for the instances used in this Demo. 

In [20]:
sm_client.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': '30da273b-5917-4cab-bc60-32b72ea48ba1',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '30da273b-5917-4cab-bc60-32b72ea48ba1',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Thu, 16 Jul 2026 16:56:21 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

You may also want to delete any other resources you might have created such as SNS topics, S3 objects, etc.

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-async_inference_walkthrough|sm-async_inference_walkthrough.ipynb)
